# Listed Transactions Analysis
## Step 1: Imported Libraries

In [ ]:
import pandas as pd
import glob
import os

## Step 2: Loaded Listed CSV Files

In [ ]:
listed_files = glob.glob("../data/CRMLSListing*.csv")
print(len(listed_files))
print(listed_files)

## Step 3: Explored a Single File
Inspected one file before combining — checked shape, columns, data types, and sample rows.

In [ ]:
df_sample = pd.read_csv(listed_files[0], low_memory=False, encoding="latin-1")
print(df_sample.shape)
print(df_sample.columns.tolist())
print(df_sample.dtypes)
print(df_sample.head(3))

## Step 4: Combined All 27 Listed Files
Stacked all monthly CSVs (Jan 2024 - Mar 2026) into one DataFrame.

In [ ]:
dfs = [pd.read_csv(f, low_memory=False, encoding="latin-1") for f in listed_files]
df_listed = pd.concat(dfs, ignore_index=True)
print(df_listed.shape)
print(df_listed.columns.tolist())

## Step 5: Explored the Combined Dataset
Checked null counts and unique values in key categorical fields.

In [ ]:
print(df_listed.isnull().sum().sort_values(ascending=False))
print(df_listed["PropertyType"].unique())

In [ ]:
critical_cols = ["ListPrice","OriginalListPrice","LivingArea","DaysOnMarket","PropertyType","ListingContractDate","City","PostalCode","BedroomsTotal","BathroomsTotalInteger","MlsStatus"]
print(df_listed[critical_cols].isnull().sum())

## Step 6: Cleaned the Data
- Filtered to Residential only
- Dropped rows missing ListPrice or LivingArea
- Dropped duplicate .1 columns from API script
- Converted date columns to datetime
- Dropped columns that were mostly null or irrelevant

In [ ]:
df_listed = df_listed[df_listed["PropertyType"] == "Residential"]
df_listed = df_listed.dropna(subset=["ListPrice", "LivingArea"])
print(df_listed.shape)

In [ ]:
df_listed["ListingContractDate"] = pd.to_datetime(df_listed["ListingContractDate"])
df_listed["ContractStatusChangeDate"] = pd.to_datetime(df_listed["ContractStatusChangeDate"])
df_listed["PurchaseContractDate"] = pd.to_datetime(df_listed["PurchaseContractDate"])
df_listed["CloseDate"] = pd.to_datetime(df_listed["CloseDate"])
print(df_listed[["ListingContractDate","ContractStatusChangeDate","PurchaseContractDate","CloseDate"]].dtypes)

In [ ]:
cols_to_drop = ["PropertyType.1","ListAgentFirstName.1","DaysOnMarket.1","LivingArea.1","Longitude.1","Latitude.1","ListPrice.1","ListAgentLastName.1","CloseDate.1","BuyerOfficeName.1","UnparsedAddress.1","BuyerAgencyCompensationType","BuyerAgencyCompensation","BusinessType","TaxYear","AboveGradeFinishedArea","MiddleOrJuniorSchoolDistrict"]
cols_to_drop = [c for c in cols_to_drop if c in df_listed.columns]
df_listed = df_listed.drop(columns=cols_to_drop)
print(df_listed.shape)

In [ ]:
df_listed.isnull().sum().sort_values(ascending=False)

## Step 7: Removed Outliers Using IQR
Applied IQR method to ListPrice, LivingArea, and DaysOnMarket.

In [ ]:
for col in ["ListPrice", "LivingArea", "DaysOnMarket"]:
    Q1 = df_listed[col].quantile(0.25)
    Q3 = df_listed[col].quantile(0.75)
    IQR = Q3 - Q1
    df_listed = df_listed[(df_listed[col] >= Q1 - 1.5*IQR) & (df_listed[col] <= Q3 + 1.5*IQR)]
print(df_listed.shape)

## Step 8: Feature Engineering
Created new calculated columns: ListPricePerSqFt, PriceRatio, ListYear, ListMonth, YrMo.

In [ ]:
df_listed["ListPricePerSqFt"] = df_listed["ListPrice"] / df_listed["LivingArea"]
df_listed["PriceRatio"] = df_listed["ClosePrice"] / df_listed["ListPrice"]
df_listed["ListYear"] = df_listed["ListingContractDate"].dt.year
df_listed["ListMonth"] = df_listed["ListingContractDate"].dt.month
df_listed["YrMo"] = df_listed["ListingContractDate"].dt.to_period("M").astype(str)
print(df_listed[["ListPrice","LivingArea","ListPricePerSqFt","PriceRatio","ListYear","ListMonth","YrMo"]].head(3))

## Step 9: Exported Clean CSV

In [ ]:
df_listed.to_csv("../data/listed_clean.csv", index=False)
print("Exported:", df_listed.shape)